In [1]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

   ---------------------------------------- 0.0/730.3 kB ? eta -:--:--
   ---------------------------------------- 730.3/730.3 kB 15.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 22.4 MB/s  0:00:00

   ---------------------------------------- 0/9 [flatbuffers]
   ---- ----------------------------------- 1/9 [wassima]
   -------- ------------------------------- 2/9 [qh3]
   -------- ------------------------------- 2/9 [qh3]
   -------- ------------------------------- 2/9 [qh3]
   ------------- -------------------------- 3/9 [openmeteo-sdk]
   ----------------- ---------------------- 4/9 [jh2]
   ----------------- ---------------------- 4/9 [jh2]
   ----------------- ---------------------- 4/9 [jh2]
   ---------------------- ----------------- 5/9 [h11]
   ---------------------- ----------------- 5/9 [h11]
   -------------------------- ------------- 6/9 [urllib3-future]
   -----------------------

In [19]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 15.975753117352811,
	"longitude": 108.19961460600025,
	"hourly": ["temperature_2m", "precipitation_probability", "precipitation", "uv_index", "wind_speed_10m", "visibility", "relative_humidity_2m", "surface_pressure", "weather_code"],
	"past_days": 0,
	"forecast_days": 14,
    "timezone": "auto"
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
timezone_name = response.Timezone()
timezone_abbr = response.TimezoneAbbreviation()
utc_offset_seconds = response.UtcOffsetSeconds()

if isinstance(timezone_name, bytes):
    timezone_name = timezone_name.decode("utf-8")
if isinstance(timezone_abbr, bytes):
    timezone_abbr = timezone_abbr.decode("utf-8")

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {timezone_name} ({timezone_abbr})")
print(f"Timezone difference to GMT+0: {utc_offset_seconds}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_uv_index = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
hourly_visibility = hourly.Variables(5).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(8).ValuesAsNumpy()

start_utc = pd.to_datetime(hourly.Time(), unit = "s", utc = True)
end_utc = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True)
freq = pd.Timedelta(seconds = hourly.Interval())

hourly_data = {
    "date_local": pd.date_range(
        start = start_utc.tz_convert(timezone_name),
        end = end_utc.tz_convert(timezone_name),
        freq = freq,
        inclusive = "left"
    ),
    "date_utc": pd.date_range(
        start = start_utc,
        end = end_utc,
        freq = freq,
        inclusive = "left"
    )
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["uv_index"] = hourly_uv_index
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["visibility"] = hourly_visibility
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["weather_code"] = hourly_weather_code

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 16.0°N 108.125°E
Elevation: 5.0 m asl
Timezone: Asia/Ho_Chi_Minh (GMT+7)
Timezone difference to GMT+0: 25200s

Hourly data
                    date_local                  date_utc  temperature_2m  \
0   2026-04-23 00:00:00+07:00 2026-04-22 17:00:00+00:00       25.271000   
1   2026-04-23 01:00:00+07:00 2026-04-22 18:00:00+00:00       25.021000   
2   2026-04-23 02:00:00+07:00 2026-04-22 19:00:00+00:00       24.971001   
3   2026-04-23 03:00:00+07:00 2026-04-22 20:00:00+00:00       24.921000   
4   2026-04-23 04:00:00+07:00 2026-04-22 21:00:00+00:00       24.871000   
..                        ...                       ...             ...   
331 2026-05-06 19:00:00+07:00 2026-05-06 12:00:00+00:00       26.996000   
332 2026-05-06 20:00:00+07:00 2026-05-06 13:00:00+00:00       26.046000   
333 2026-05-06 21:00:00+07:00 2026-05-06 14:00:00+00:00       25.246000   
334 2026-05-06 22:00:00+07:00 2026-05-06 15:00:00+00:00       24.796000   
335 2026-05-06 23:00:00+07:00 2026-05-

In [ ]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": 15.975753117352811,
	"longitude": 108.19961460600025,
	"hourly": "us_aqi",
	"past_days": 0,
	"forecast_days": 5,
    "timezone": "auto"
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
timezone_name = response.Timezone()
timezone_abbr = response.TimezoneAbbreviation()
utc_offset_seconds = response.UtcOffsetSeconds()

if isinstance(timezone_name, bytes):
    timezone_name = timezone_name.decode("utf-8")
if isinstance(timezone_abbr, bytes):
    timezone_abbr = timezone_abbr.decode("utf-8")

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {timezone_name} ({timezone_abbr})")
print(f"Timezone difference to GMT+0: {utc_offset_seconds}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_us_aqi = hourly.Variables(0).ValuesAsNumpy()

start_utc = pd.to_datetime(hourly.Time(), unit = "s", utc = True)
end_utc = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True)
freq = pd.Timedelta(seconds = hourly.Interval())

hourly_data = {
    "date_local": pd.date_range(
        start = start_utc.tz_convert(timezone_name),
        end = end_utc.tz_convert(timezone_name),
        freq = freq,
        inclusive = "left"
    ),
    "date_utc": pd.date_range(
        start = start_utc,
        end = end_utc,
        freq = freq,
        inclusive = "left"
    )
}

hourly_data["us_aqi"] = hourly_us_aqi

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 16.0°N 108.20001220703125°E
Elevation: 5.0 m asl
Timezone difference to GMT+0: 25200s

Hourly data
                          date      us_aqi
0   2026-04-22 17:00:00+00:00  126.885429
1   2026-04-22 18:00:00+00:00  126.364586
2   2026-04-22 19:00:00+00:00  126.281250
3   2026-04-22 20:00:00+00:00  126.500008
4   2026-04-22 21:00:00+00:00  126.781250
..                        ...         ...
115 2026-04-27 12:00:00+00:00  118.154755
116 2026-04-27 13:00:00+00:00  101.573135
117 2026-04-27 14:00:00+00:00   94.982269
118 2026-04-27 15:00:00+00:00   97.269508
119 2026-04-27 16:00:00+00:00   99.716316

[120 rows x 2 columns]


In [21]:
from pathlib import Path
import json
import pandas as pd

if "hourly_dataframe" not in globals():
    raise RuntimeError("Run the weather forecast cell first so hourly_dataframe is available.")

timezone_name_value = globals().get("timezone_name")
timezone_abbr_value = globals().get("timezone_abbr")
if isinstance(timezone_name_value, bytes):
    timezone_name_value = timezone_name_value.decode("utf-8")
if isinstance(timezone_abbr_value, bytes):
    timezone_abbr_value = timezone_abbr_value.decode("utf-8")

export_df = hourly_dataframe.copy()

# Serialize all datetime columns to ISO 8601 strings (keep timezone info when present).
for col in export_df.columns:
    if pd.api.types.is_datetime64_any_dtype(export_df[col]):
        if export_df[col].dt.tz is not None:
            export_df[col] = export_df[col].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
            export_df[col] = export_df[col].str.replace(r"([+-]\d{2})(\d{2})$", r"\1:\2", regex=True)
        else:
            export_df[col] = export_df[col].dt.strftime("%Y-%m-%dT%H:%M:%S")

# Convert NaN to None so JSON is valid and easier to consume.
export_df = export_df.where(pd.notna(export_df), None)

payload = {
    "source": "open-meteo",
    "timezone": timezone_name_value,
    "timezone_abbreviation": timezone_abbr_value,
    "utc_offset_seconds": globals().get("utc_offset_seconds"),
    "count": int(len(export_df)),
    "fields": export_df.columns.tolist(),
    "values": export_df.to_dict(orient="records"),
}

output_path = Path("hourly_values.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(f"Saved: {output_path.resolve()}")
print(f"Total rows: {len(export_df)}")
print("Fields:", ", ".join(payload["fields"]))
print(f"Timezone in payload: {payload['timezone']} ({payload['timezone_abbreviation']})")
print(export_df.to_string(index=False))

Saved: D:\Work\Learn\HK6_2025-2026\flutter\Search_Wheather_forecast\maps\forecast_14_days\hourly_values.json
Total rows: 336
Fields: date_local, date_utc, temperature_2m, precipitation_probability, precipitation, uv_index, wind_speed_10m, visibility, relative_humidity_2m, surface_pressure, weather_code
Timezone in payload: Asia/Ho_Chi_Minh (GMT+7)
               date_local                  date_utc  temperature_2m  precipitation_probability  precipitation  uv_index  wind_speed_10m  visibility  relative_humidity_2m  surface_pressure  weather_code
2026-04-23T00:00:00+07:00 2026-04-22T17:00:00+00:00       25.271000                       0.00          0.000      0.00        5.771239     24140.0                  84.0       1007.523132           2.0
2026-04-23T01:00:00+07:00 2026-04-22T18:00:00+00:00       25.021000                       0.00          0.000      0.00        5.052841     24140.0                  87.0       1007.023132           2.0
2026-04-23T02:00:00+07:00 2026-04-22T19:00:0

In [3]:
import requests
import json

lat = 15.9757
lon = 108.1996
url = "https://api.open-meteo.com/v1/forecast"

# Thêm các biến cơ bản để đảm bảo API hoạt động
params = {
    "latitude": lat,
    "longitude": lon,
    "daily": ["temperature_2m_max", "moon_phase", "uv_index_max"], # Truyền dạng list cho chắc chắn
    "timezone": "Asia/Bangkok",
    "forecast_days": 14
}

response = requests.get(url, params=params)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print("--- KẾT QUẢ ---")
    print(json.dumps(data['daily'], indent=2))
else:
    # Đây là phần quan trọng để biết tại sao lỗi
    print("--- THÔNG BÁO LỖI TỪ SERVER ---")
    print(response.text)

Status Code: 400
--- THÔNG BÁO LỖI TỪ SERVER ---
{"error":true,"reason":"Data corrupted at path ''. Cannot initialize ForecastVariableDaily from invalid String value moon_phase."}
